# Bronze Raw Data Generator (CSV)

Generates synthetic raw source data (items, orders, products, suppliers, customers) for the upwind_demo bronze/silver/gold pipeline.

Each table is generated in **SQL** into a temp view, then exported as CSV files to the Unity Catalog volume `raw_csv` (created by the bundle deploy under `${var.catalog}.${var.schema}`). The job passes the volume path as the `volume_base` notebook parameter; the notebook has a dev default for interactive runs.

In [0]:
%sql
-- Order items: 900 rows across 300 orders
CREATE OR REPLACE TEMP VIEW v_raw_items AS
SELECT item_id,
       cast(rand() * 300 AS INT) + 1 AS order_id,
       cast(rand() * 50  AS INT) + 1 AS product_id,
       cast(rand() * 10  AS INT) + 1 AS quantity,
       round(rand() * 495 + 5, 2)    AS unit_price
FROM range(1, 901) t(item_id);

SELECT * FROM v_raw_items LIMIT 5;

In [0]:
%sql
-- Orders: 300 rows
CREATE OR REPLACE TEMP VIEW v_raw_orders AS
SELECT o AS order_id,
       cast(rand() * 100 AS INT) + 1 AS customer_id,
       date_format(date_add(DATE'2025-01-01', cast(rand() * 178 AS INT)), 'yyyy-MM-dd') AS order_date,
       element_at(array('completed', 'pending', 'cancelled', 'shipped'), cast(rand() * 4 AS INT) + 1) AS status
FROM range(1, 301) t(o);

SELECT * FROM v_raw_orders LIMIT 5;

In [0]:
%sql
-- Products: 50 rows
CREATE OR REPLACE TEMP VIEW v_raw_products AS
SELECT id AS product_id,
       concat('Product_', id) AS product_name,
       element_at(array('Electronics', 'Home', 'Sports', 'Toys', 'Books'), cast(rand() * 5 AS INT) + 1) AS category,
       round(rand() * 495 + 5, 2) AS price,
       cast(rand() * 20 AS INT) + 1 AS supplier_id
FROM range(1, 51) t(id);

SELECT * FROM v_raw_products LIMIT 5;

In [0]:
%sql
-- Suppliers: 20 rows
CREATE OR REPLACE TEMP VIEW v_raw_suppliers AS
SELECT id AS supplier_id,
       concat('Supplier_', id) AS supplier_name,
       element_at(array('Israel', 'USA', 'Germany', 'China', 'Turkey'), cast(rand() * 5 AS INT) + 1) AS country,
       round(rand() * 3.9 + 1, 1) AS rating
FROM range(1, 21) t(id);

SELECT * FROM v_raw_suppliers LIMIT 5;

In [0]:
%sql
-- Customers: 100 rows (email is null for ~5% to simulate dirty raw data)
CREATE OR REPLACE TEMP VIEW v_raw_customers AS
SELECT id AS customer_id,
       concat('Customer_', id) AS customer_name,
       CASE WHEN rand() < 0.05 THEN NULL ELSE concat('customer', id, '@example.com') END AS email,
       element_at(array('Tel Aviv', 'Haifa', 'Jerusalem', 'Netanya', 'Beersheba'), cast(rand() * 5 AS INT) + 1) AS city,
       date_format(date_add(DATE'2024-01-01', cast(rand() * 364 AS INT)), 'yyyy-MM-dd') AS signup_date
FROM range(1, 101) t(id);

SELECT * FROM v_raw_customers LIMIT 5;

In [0]:
# Export all generated views as CSV files to the bronze volume.
# The job task passes the per-target volume path via base_parameters;
# the default below works for interactive runs on the dev catalog.
dbutils.widgets.text("volume_base", "/Volumes/ofer_demo_dev/bronze/raw_csv")
VOLUME_BASE = dbutils.widgets.get("volume_base")

views = {
    "raw_items": "v_raw_items",
    "raw_orders": "v_raw_orders",
    "raw_products": "v_raw_products",
    "raw_suppliers": "v_raw_suppliers",
    "raw_customers": "v_raw_customers",
}

for table_name, view_name in views.items():
    df = spark.table(view_name)
    (
        df.write
          .mode("overwrite")
          .option("header", "true")
          .csv(f"{VOLUME_BASE}/{table_name}")
    )
    print(f"{table_name}: {df.count()} rows -> {VOLUME_BASE}/{table_name}")